In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [6]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [7]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [8]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return len(self.features)

  def __getitem__(self, idx):

    return self.features[idx], self.labels[idx]



In [9]:
train_dataset = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

In [10]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

In [11]:
import torch.nn as nn


class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)

    return out

In [12]:
learning_rate = 0.1
epochs = 25

In [13]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()

In [14]:
for epoch in range(epochs):
    for batch_features,batch_labels in train_loader:
        y_pred = model(batch_features)
        loss = loss_function(y_pred,batch_labels.view(-1,1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch: {epoch+1},Loss: {loss.item()}')

Epoch: 1,Loss: 0.2427644282579422
Epoch: 2,Loss: 0.13163962960243225
Epoch: 3,Loss: 0.07783247530460358
Epoch: 4,Loss: 0.06453545391559601
Epoch: 5,Loss: 0.5718398094177246
Epoch: 6,Loss: 0.029727669432759285
Epoch: 7,Loss: 0.3785417973995209
Epoch: 8,Loss: 0.1173362135887146
Epoch: 9,Loss: 0.10275837033987045
Epoch: 10,Loss: 0.030118102207779884
Epoch: 11,Loss: 0.05756503343582153
Epoch: 12,Loss: 0.030960584059357643
Epoch: 13,Loss: 0.008758582174777985
Epoch: 14,Loss: 0.017065580934286118
Epoch: 15,Loss: 0.038037486374378204
Epoch: 16,Loss: 0.008541779592633247
Epoch: 17,Loss: 0.01818576268851757
Epoch: 18,Loss: 0.002309805015102029
Epoch: 19,Loss: 0.0072859348729252815
Epoch: 20,Loss: 0.1561501920223236
Epoch: 21,Loss: 0.019746171310544014
Epoch: 22,Loss: 0.1372884213924408
Epoch: 23,Loss: 0.007632753811776638
Epoch: 24,Loss: 0.08857178688049316
Epoch: 25,Loss: 0.02030148170888424


In [16]:
model.eval()
accuracy_list = []

with torch.no_grad():
    for batch_features,batch_labels in test_loader:
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()

        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean()
        accuracy_list.append(batch_accuracy)

oveall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(oveall_accuracy)

tensor(0.9531)
